In [ ]:
import duckdb
p = r"..."
conn = duckdb.connect(p)

In [ ]:
schema = ('acct_num', 'part_number', 'qty', 'amount', 'invoice_date', 'part_category', 'year')

In [ ]:
q = """
WITH filtered as (
    SELECT
        acct_num, 
        part_number,
        amount,
        year
    FROM sales s
    RIGHT JOIN customers c ON s.acct_num = c.child_acct
), 
grouped_2025 AS (
    SELECT
        acct_num, 
        part_number,
        SUM(amount) AS total_2025,
        year
    FROM filtered
    WHERE year = 2025
    GROUP BY acct_num, part_number, year
    ORDER BY acct_num
), grouped_2024 AS (
    SELECT
        acct_num, 
        part_number,
        SUM(amount) AS total_2024,
        year
    FROM filtered
    WHERE year = 2024
    GROUP BY acct_num, part_number, year
    ORDER BY acct_num
), ranked_2025 AS (
    SELECT 
        acct_num,
        part_number,
        total_2025,
        row_number() OVER (PARTITION BY acct_num ORDER BY total_2025 DESC) AS rank
    FROM grouped_2025
), rank_filtered_2025 AS (
SELECT 
    acct_num,
    part_number,
    total_2025,
    rank,
FROM ranked_2025
WHERE rank <= 10
)
SELECT 
    rf_25.acct_num,
    rf_25.part_number,
    rf_25.total_2025,
    g_24.total_2024,
    rf_25.rank
FROM rank_filtered_2025 rf_25
LEFT JOIN grouped_2024 g_24 ON rf_25.acct_num = g_24.acct_num AND rf_25.part_number = g_24.part_number
ORDER BY rf_25.acct_num, rf_25.rank 
"""

df = conn.query(query=q).df()

df

In [ ]:
"""
Month over Month; even if a value is zero
"""

import pandas as pd
dates_tbl = pd.DataFrame(columns=["dates"], data=pd.date_range(start="1/1/2024", end="12/31/2025", freq='ME'))
dates_tbl = dates_tbl.assign(Month=dates_tbl["dates"].dt.month, Year=dates_tbl["dates"].dt.year)
duckdb.sql("CREATE OR REPLACE TABLE dates_tbl AS SELECT * FROM dates_tbl")
duckdb.sql("INSERT INTO dates_tbl SELECT * FROM dates_tbl")
dates_q = """SELECT * FROM dates_tbl"""
dates_sql_tbl = conn.query(query=dates_q).df()
dates_sql_tbl


,dates,Month,Year
0,2024-01-31,1,2024
1,2024-02-29,2,2024
2,2024-03-31,3,2024
3,2024-04-30,4,2024
4,2024-05-31,5,2024
5,2024-06-30,6,2024
6,2024-07-31,7,2024
7,2024-08-31,8,2024
8,2024-09-30,9,2024
9,2024-10-31,10,2024


In [63]:
q = """
WITH grouped_sales AS (
  SELECT 
    acct_num,
    ROUND(SUM(amount), 2) as month_net,
    EXTRACT(Month FROM invoice_date) as invoice_month,
    EXTRACT(Year FROM invoice_date) as invoice_year
  FROM sales
  WHERE acct_num = 'STA810401'
  GROUP BY acct_num, invoice_month, invoice_year
  ),
dates AS (
  SELECT 
    month,
    year
  FROM dates_tbl
)
SELECT
  d.month,
  d.year,
  gs.acct_num,
  gs.month_net,
  gs.invoice_month,
  gs.invoice_year
FROM dates d
LEFT JOIN grouped_sales gs ON gs.invoice_month = d.month AND gs.invoice_year = d.year
ORDER BY invoice_year, invoice_month
"""

df = conn.query(query=q).df()

df


,Month,Year,acct_num,month_net,invoice_month,invoice_year
0,1,2024,STA810401,19019.56,1,2024
1,2,2024,STA810401,33282.50,2,2024
2,3,2024,STA810401,11227.01,3,2024
3,4,2024,STA810401,28317.24,4,2024
4,5,2024,STA810401,8990.80,5,2024
5,6,2024,STA810401,18849.48,6,2024
6,7,2024,STA810401,14175.32,7,2024
7,8,2024,STA810401,23065.68,8,2024
8,9,2024,STA810401,9935.34,9,2024
9,10,2024,STA810401,27952.11,10,2024
